# Promote Champion
Rama **TRUE** del health gate. Despliega la versión challenger al 100% del tráfico como served entity `champion` del endpoint y mueve el alias `champion` de UC a esta versión. Idempotente.

In [0]:
%pip install -q databricks-sdk mlflow

In [0]:
import mlflow
from mlflow.tracking import MlflowClient
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import serving
mlflow.set_registry_uri("databricks-uc")

MODEL_NAME = dbutils.widgets.get("model_name")
MODEL_VERSION = dbutils.widgets.get("model_version")
ENDPOINT_NAME = dbutils.widgets.get("endpoint_name")

client = MlflowClient()
w = WorkspaceClient()

# 100% a la nueva version, como served entity "champion"
served = [serving.ServedEntityInput(name="champion", entity_name=MODEL_NAME,
                                   entity_version=str(MODEL_VERSION),
                                   workload_size="Small", scale_to_zero_enabled=True)]
routes = [serving.Route(served_model_name="champion", traffic_percentage="100")]
w.serving_endpoints.update_config(name=ENDPOINT_NAME, served_entities=served,
                                  traffic_config=serving.TrafficConfig(routes=routes))

# mover el alias champion de UC a la nueva version (idempotente) y quitar challenger
client.set_registered_model_alias(MODEL_NAME, "champion", MODEL_VERSION)
try:
    client.delete_registered_model_alias(MODEL_NAME, "challenger")
except Exception:
    pass

print(f"{MODEL_NAME} v{MODEL_VERSION} promovido a champion (endpoint 100%, alias champion)")